# MAG7 Bull/Bear Rule Mining - Alternative Time Split

This notebook reruns the rule-mining stage with a later, larger held-out test window.

Main design:

- Discovery / rule mining: 2023-01-01 to 2023-12-31
- Validation / rule selection: 2024-01-01 to 2024-12-31
- Test: 2025-01-01 to 2026-06-01

This keeps all model selection inside 2023-2024, then tests on 2025 through June 1st 2026.

The notebook reuses the existing strict-event outputs and the existing LLM refinement checkpoint. It does not rerun the expensive LLM extraction.

## What to look for

You are checking whether the original conclusion survives a different held-out period.

Focus on:

- Does `rule_matched` stay positive on the 2025-2026 test period?
- Is `rule_matched` better than `llm_expected_label` and `all_events_baseline`?
- How many test events are rule-matched? A bigger test set is good, but only if the matched events are economically clean.
- Are the selected rules similar to the old 10 rules, or does the system find a totally different story?

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import types

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "scripts" / "run_mag7_bull_bear_rule_mining.py").exists():
    PROJECT = Path.cwd() / "Sentiment project"

STRICT_DIR = PROJECT / "outputs" / "1_mag7_strict_event_full"
ORIGINAL_RULE_DIR = PROJECT / "outputs" / "2_bull_bear_mining"
OUT_DIR = PROJECT / "outputs" / "6_mag7_train_2023_2024_test_2025_2026_split"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The script imports transformers for the expensive LLM stage. This notebook does not use that stage,
# so allow rule-function imports even in a lightweight environment without transformers installed.
try:
    import transformers  # noqa: F401
except ModuleNotFoundError:
    transformers_stub = types.ModuleType("transformers")
    transformers_stub.AutoModelForCausalLM = object
    transformers_stub.AutoTokenizer = object
    sys.modules["transformers"] = transformers_stub

SCRIPT_PATH = PROJECT / "scripts" / "run_mag7_bull_bear_rule_mining.py"
spec = importlib.util.spec_from_file_location("mag7_rules", SCRIPT_PATH)
rules = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rules)

print("Project:", PROJECT)
print("Strict event outputs:", STRICT_DIR)
print("Original rule outputs:", ORIGINAL_RULE_DIR)
print("New split outputs:", OUT_DIR)

## Load fixed event and LLM refinement data

This is deliberately using the same strict-event data and same second-pass LLM interpretations as the original successful notebook. Only the time split and rule mining are changed.

In [ ]:
PRIMARY_HORIZON = 3
RETURN_THRESHOLD = 0.005
MIN_DISCOVERY_EVENTS = 6
MIN_VALIDATION_EVENTS = 3
LTN_EPOCHS = 800
LTN_LR = 0.05
LTN_MIN_WEIGHT = 0.60

DISCOVERY_START = "2023-01-01"
DISCOVERY_END = "2023-12-31"
VALIDATION_START = "2024-01-01"
VALIDATION_END = "2024-12-31"
TEST_START = "2025-01-01"
TEST_END = "2026-06-01"

returns = pd.read_parquet(STRICT_DIR / "event_returns.parquet")
refinements = pd.read_parquet(ORIGINAL_RULE_DIR / "event_refinement_checkpoint.parquet")
study = pd.read_parquet(ORIGINAL_RULE_DIR / "refined_event_returns.parquet")

defaults = {
    "include_candidate": False,
    "expected_label": "neutral",
    "event_subtype": "other",
    "valuation_channel": "unclear",
    "materiality_strength": "low",
    "novelty": "unclear",
    "rationale": "",
}
for col, value in defaults.items():
    study[col] = study[col].fillna(value)

study["time_published"] = pd.to_datetime(study["time_published"], utc=True)
study["event_date"] = pd.to_datetime(study["event_date"])
study["realised_label"] = study["abnormal_return"].map(lambda x: rules.realised_label(float(x), RETURN_THRESHOLD))

primary = study.loc[study["horizon_days"].eq(PRIMARY_HORIZON)].copy()

print("Raw return rows:", len(returns))
print("Merged refined return rows:", len(study))
print("Unique refined events:", refinements["event_id"].nunique())
print("Primary horizon rows:", len(primary))

## Build the new periods

This is the main robustness split. The test period is now much larger than the original 2026-only test window.

In [ ]:
def period(df, start, end):
    return df.loc[rules.period_filter(df, start, end)].copy()

periods = {
    "discovery": period(primary, DISCOVERY_START, DISCOVERY_END),
    "validation": period(primary, VALIDATION_START, VALIDATION_END),
    "test": period(primary, TEST_START, TEST_END),
}

period_summary = []
for name, df in periods.items():
    period_summary.append({
        "period": name,
        "start": {"discovery": DISCOVERY_START, "validation": VALIDATION_START, "test": TEST_START}[name],
        "end": {"discovery": DISCOVERY_END, "validation": VALIDATION_END, "test": TEST_END}[name],
        "rows": len(df),
        "include_candidates": int(df["include_candidate"].sum()),
        "bullish_realised": int(df["realised_label"].eq("bullish").sum()),
        "bearish_realised": int(df["realised_label"].eq("bearish").sum()),
        "neutral_realised": int(df["realised_label"].eq("neutral").sum()),
    })

pd.DataFrame(period_summary)

## Mine rules on 2023, validate on 2024

This is the cleanest design: 2025-2026 is not touched until the final test. If this works, it is much stronger evidence than choosing rules using 2025 data.

In [ ]:
discovery_rules = rules.mine_rules(
    periods["discovery"],
    min_n=MIN_DISCOVERY_EVENTS,
    epochs=LTN_EPOCHS,
    lr=LTN_LR,
    min_weight=LTN_MIN_WEIGHT,
)

validation_eval = rules.evaluate_rules(periods["validation"], discovery_rules, "validation", min_n=1)
selected_rules = rules.select_validated_rules(discovery_rules, validation_eval, MIN_VALIDATION_EVENTS)
test_eval = rules.evaluate_rules(periods["test"], selected_rules, "test", min_n=1)

print("Candidate rules from 2023:", len(discovery_rules))
print("Selected rules after 2024 validation:", len(selected_rules))
display(selected_rules[[
    "condition", "rule_label", "rule_score", "discovery_n", "discovery_accuracy_pct",
    "discovery_aligned_mean_pct", "validation_n", "validation_accuracy_pct", "validation_aligned_mean_pct",
]].head(30))

## Final test on 2025 to June 1st 2026

This is the key table. Compare the `rule_matched` row against `llm_expected_label` and `all_events_baseline`.

A strong result would have:

- positive `test_aligned_mean_pct`
- `test_accuracy_pct` above the LLM expected-label baseline
- enough `test_n` that it is not just a tiny lucky group
- a sign-flip p-value that is at least directionally supportive

In [ ]:
summary = {
    "discovery": rules.rule_system_summary(periods["discovery"], selected_rules, "discovery"),
    "validation": rules.rule_system_summary(periods["validation"], selected_rules, "validation"),
    "test": rules.rule_system_summary(periods["test"], selected_rules, "test"),
}

display(summary["test"])

test_lookup = summary["test"].set_index("group")
if "rule_matched" in test_lookup.index and "llm_expected_label" in test_lookup.index:
    lift = test_lookup.loc["rule_matched", "test_aligned_mean_pct"] - test_lookup.loc["llm_expected_label", "test_aligned_mean_pct"]
    acc_lift = test_lookup.loc["rule_matched", "test_accuracy_pct"] - test_lookup.loc["llm_expected_label", "test_accuracy_pct"]
    print(f"Rule-matched aligned mean lift over LLM expected label: {lift:.3f} percentage points")
    print(f"Rule-matched accuracy lift over LLM expected label: {acc_lift:.3f} percentage points")

## Matched test events to inspect

Manually read these rows. If the result rests on rule-matched events, they need to be economically direct and ticker-relevant.

Suggested labels:

- clean direct event
- indirect/weak event
- wrong ticker relevance
- plausible but noisy
- duplicate/repeat/market reaction

In [ ]:
test_applied = rules.apply_rules(periods["test"], selected_rules)
matched_test = test_applied.loc[test_applied["rule_prediction"].ne("no_rule")].copy()

audit_cols = [
    "ticker", "event_date", "time_published", "representative_title",
    "event_type", "direction", "event_subtype", "valuation_channel",
    "materiality_strength", "novelty", "include_candidate", "expected_label",
    "rule_prediction", "matched_rule", "realised_label", "abnormal_return", "rationale",
]

manual_cols = ["manual_audit_label", "manual_direct_company_mechanism", "manual_notes"]
matched_audit = matched_test[audit_cols].sort_values(["event_date", "ticker"]).copy()
for col in manual_cols:
    matched_audit[col] = ""

print("Rule-matched test events:", len(matched_audit))
display(matched_audit)

## Rule support across periods

This checks whether selected rules are supported by a reasonable number of events or whether they are tiny groups that may be unstable.

In [ ]:
support_rows = []
for period_name, df in periods.items():
    for rule in selected_rules.itertuples(index=False):
        mask = rules.match_condition(df, rule.condition)
        subset = df.loc[mask]
        support_rows.append({
            "period": period_name,
            "condition": rule.condition,
            "rule_label": rule.rule_label,
            "n": len(subset),
            "accuracy_pct": rules.summarise_predictions(subset.assign(rule_label=rule.rule_label), "rule_label")["accuracy_pct"] if len(subset) else np.nan,
            "aligned_mean_pct": rules.summarise_predictions(subset.assign(rule_label=rule.rule_label), "rule_label")["aligned_mean_pct"] if len(subset) else np.nan,
        })

support = pd.DataFrame(support_rows)
display(support.pivot_table(index=["condition", "rule_label"], columns="period", values="n", aggfunc="sum", fill_value=0).reset_index())
display(support)

## Save split outputs

This saves the tables from this alternative split so you can cite them or inspect them later.

In [ ]:
discovery_rules.to_csv(OUT_DIR / "candidate_rules_discovery_2023.csv", index=False)
validation_eval.to_csv(OUT_DIR / "candidate_rules_validation_2024.csv", index=False)
selected_rules.to_csv(OUT_DIR / "selected_validated_rules_2023_2024.csv", index=False)
test_eval.to_csv(OUT_DIR / "selected_rules_test_2025_to_2026_06_01.csv", index=False)
summary["discovery"].to_csv(OUT_DIR / "rule_system_discovery_2023_summary.csv", index=False)
summary["validation"].to_csv(OUT_DIR / "rule_system_validation_2024_summary.csv", index=False)
summary["test"].to_csv(OUT_DIR / "rule_system_test_2025_to_2026_06_01_summary.csv", index=False)
matched_audit.to_csv(OUT_DIR / "manual_audit_rule_matched_test_2025_to_2026_06_01.csv", index=False)

manifest = {
    "design": "discovery_2023_validation_2024_test_2025_to_2026_06_01",
    "input_strict_dir": str(STRICT_DIR),
    "input_refinement_dir": str(ORIGINAL_RULE_DIR),
    "primary_horizon": PRIMARY_HORIZON,
    "return_threshold": RETURN_THRESHOLD,
    "discovery": [DISCOVERY_START, DISCOVERY_END],
    "validation": [VALIDATION_START, VALIDATION_END],
    "test": [TEST_START, TEST_END],
    "min_discovery_events": MIN_DISCOVERY_EVENTS,
    "min_validation_events": MIN_VALIDATION_EVENTS,
    "n_candidate_rules": int(len(discovery_rules)),
    "n_selected_rules": int(len(selected_rules)),
    "n_rule_matched_test": int(len(matched_audit)),
}
(OUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved outputs to", OUT_DIR)

## Exploratory variant: mine rules using all 2023-2024, test on 2025-2026

This variant uses the whole 2023-2024 period for discovery. It is useful, but it is less conservative because there is no separate validation year. Treat this as exploratory, not the main dissertation result.

What to look for:

- If this works but the validated design above fails, the rule system may need more training data but is less clean statistically.
- If both fail, the old result is probably sensitive to the original split.
- If both work, that is much stronger evidence.

In [ ]:
train_2023_2024 = period(primary, "2023-01-01", "2024-12-31")
test_2025_2026 = period(primary, TEST_START, TEST_END)

all_train_rules = rules.mine_rules(
    train_2023_2024,
    min_n=MIN_DISCOVERY_EVENTS,
    epochs=LTN_EPOCHS,
    lr=LTN_LR,
    min_weight=LTN_MIN_WEIGHT,
)

all_train_test_summary = rules.rule_system_summary(test_2025_2026, all_train_rules, "test")
all_train_test_applied = rules.apply_rules(test_2025_2026, all_train_rules)
all_train_matched = all_train_test_applied.loc[all_train_test_applied["rule_prediction"].ne("no_rule")].copy()

print("Exploratory all-2023-2024 candidate rules:", len(all_train_rules))
display(all_train_rules[["condition", "rule_label", "rule_score", "discovery_n", "discovery_accuracy_pct", "discovery_aligned_mean_pct"]].head(30))
display(all_train_test_summary)
print("Exploratory rule-matched test events:", len(all_train_matched))

In [ ]:
all_train_rules.to_csv(OUT_DIR / "exploratory_all_2023_2024_candidate_rules.csv", index=False)
all_train_test_summary.to_csv(OUT_DIR / "exploratory_all_2023_2024_test_2025_to_2026_06_01_summary.csv", index=False)
all_train_matched[audit_cols].sort_values(["event_date", "ticker"]).to_csv(
    OUT_DIR / "exploratory_all_2023_2024_rule_matched_test_events.csv",
    index=False,
)
print("Saved exploratory outputs to", OUT_DIR)

## Interpretation guide

Use the main split as your primary robustness check.

Possible conclusions:

- Strong: rule-matched test performance remains positive, beats LLM expected-label baseline, and matched events are mostly clean direct mechanisms.
- Mixed: rule-matched performance is positive but depends on a small number of events or weak/indirect mechanisms.
- Fragile: rule-matched performance drops, selected rules change substantially, or test matched events are mostly noisy.

Do not choose between these splits based only on which has the best p-value. Use this notebook to show whether the original finding survives a harder and more useful 2025-2026 test window.